# Cairnveil Self-RAG Notebook

Cybersecurity-focused Self-RAG implementation for Cairnveil Security. This notebook follows the Self-RAG idea from Asai et al.: decide when to retrieve, retrieve on demand, generate with evidence, and critique the result before returning it.

The sample corpus in `documents/` is fictional and designed for cybersecurity company questions about MDR, incident response, compliance support, evidence handling, and security policies.

In [2]:
# Uncomment this cell in a fresh environment.
# %pip install -r requirements.txt

In [ ]:
from __future__ import annotations

import json
import os
import re
from dataclasses import asdict, dataclass, replace
from pathlib import Path
from typing import Any, Dict, List, Literal, TypedDict

import numpy as np
from dotenv import load_dotenv
from google import genai
from google.genai import types
from langgraph.graph import END, START, StateGraph
from openai import OpenAI
from pydantic import BaseModel, Field

load_dotenv()

PROJECT_ROOT = Path.cwd()
DOCS_DIR = PROJECT_ROOT / 'documents'

KILOCODE_API_KEY = os.getenv('KILOCODE_API_KEY') or os.getenv('KILO_API_KEY')
KILO_API_BASE = os.getenv('KILO_API_BASE', 'https://api.kilo.ai/api/gateway').rstrip('/')
KILO_MODEL = os.getenv('KILO_MODEL', 'nvidia/nemotron-3-super-120b-a12b:free')
GEMINI_API_KEY = os.getenv('GEMINI_API_KEY') or os.getenv('GOOGLE_API_KEY')
GEMINI_EMBEDDING_MODEL = os.getenv('GEMINI_EMBEDDING_MODEL', 'gemini-embedding-001')

client = OpenAI(api_key=KILOCODE_API_KEY, base_url=KILO_API_BASE) if KILOCODE_API_KEY else None
embedding_client = genai.Client(api_key=GEMINI_API_KEY) if GEMINI_API_KEY else None

print(f'Documents directory: {DOCS_DIR}')
print(f'Kilo model: {KILO_MODEL}')
print(f'Gemini embedding model: {GEMINI_EMBEDDING_MODEL}')
if client is None:
    print('Set KILOCODE_API_KEY in .env before running the graph.')
if embedding_client is None:
    print('Set GEMINI_API_KEY in .env before building the vector retriever.')

## Load and Chunk Cybersecurity Documents

The retriever uses Gemini embeddings for semantic search and stores vectors in FAISS when the `faiss-cpu` package is available. It supports Markdown, text, and PDF files, then chunks them into section-aware passages.

In [11]:
@dataclass
class Chunk:
    id: str
    source: str
    section: str
    text: str
    score: float = 0.0


def read_pdf(path: Path) -> str:
    from pypdf import PdfReader

    reader = PdfReader(str(path))
    return '\n'.join(page.extract_text() or '' for page in reader.pages)


def read_document(path: Path) -> str:
    suffix = path.suffix.lower()
    if suffix == '.pdf':
        return read_pdf(path)
    if suffix in {'.md', '.markdown', '.txt'}:
        return path.read_text(encoding='utf-8')
    raise ValueError(f'Unsupported document type: {path.name}')


def split_sections(text: str, default_section: str) -> List[tuple[str, str]]:
    sections: List[tuple[str, str]] = []
    heading = default_section
    buffer: List[str] = []

    for line in text.splitlines():
        if line.startswith('#') and line.lstrip('#').strip():
            if buffer:
                sections.append((heading, '\n'.join(buffer).strip()))
                buffer = []
            heading = line.lstrip('#').strip()
        buffer.append(line)

    if buffer:
        sections.append((heading, '\n'.join(buffer).strip()))

    return [(section, body) for section, body in sections if body]


def word_chunks(text: str, max_words: int = 180, overlap: int = 35) -> List[str]:
    words = text.split()
    if not words:
        return []

    step = max_words - overlap
    chunks = []
    for start in range(0, len(words), step):
        window = words[start:start + max_words]
        if window:
            chunks.append(' '.join(window))
        if start + max_words >= len(words):
            break
    return chunks


def load_corpus(docs_dir: Path) -> List[Chunk]:
    supported = {'.md', '.markdown', '.txt', '.pdf'}
    files = sorted(path for path in docs_dir.iterdir() if path.suffix.lower() in supported)
    chunks: List[Chunk] = []

    for path in files:
        text = read_document(path)
        for section, section_text in split_sections(text, default_section=path.stem):
            for chunk_text in word_chunks(section_text):
                chunk_id = f'{path.stem}:{len(chunks):04d}'
                chunks.append(Chunk(id=chunk_id, source=path.name, section=section, text=chunk_text))

    if not chunks:
        raise RuntimeError(f'No supported documents found in {docs_dir}')
    return chunks


chunks = load_corpus(DOCS_DIR)
sources = sorted({chunk.source for chunk in chunks})
print(f'Loaded {len(chunks)} chunks from {len(sources)} documents.')
for source in sources:
    print('-', source)

Loaded 33 chunks from 4 documents.
- cairnveil_company_profile.md
- cairnveil_incident_response_playbook.md
- cairnveil_security_policies.md
- cairnveil_services_and_pricing.md


In [12]:
def normalize_vectors(vectors: np.ndarray) -> np.ndarray:
    norms = np.linalg.norm(vectors, axis=1, keepdims=True)
    norms[norms == 0] = 1.0
    return (vectors / norms).astype('float32')


def embedding_values(response) -> List[List[float]]:
    return [list(embedding.values) for embedding in response.embeddings]


class GeminiVectorRetriever:
    def __init__(self, chunks: List[Chunk], model: str = GEMINI_EMBEDDING_MODEL):
        if embedding_client is None:
            raise RuntimeError('Missing GEMINI_API_KEY. Add it to .env, then rerun the setup cell.')

        self.chunks = chunks
        self.model = model
        self.faiss = self._load_faiss()
        self.embeddings = self._embed_chunks(chunks)
        self.index = self._build_index(self.embeddings)

    @staticmethod
    def _load_faiss():
        try:
            import faiss
            return faiss
        except ImportError:
            print('faiss-cpu is not installed; using NumPy cosine search fallback.')
            return None

    @staticmethod
    def _retrieval_text(chunk: Chunk) -> str:
        return f'Source: {chunk.source}\nSection: {chunk.section}\n{chunk.text}'

    def _embed_document(self, chunk: Chunk) -> List[float]:
        response = embedding_client.models.embed_content(
            model=self.model,
            contents=self._retrieval_text(chunk),
            config=types.EmbedContentConfig(
                task_type='RETRIEVAL_DOCUMENT',
                title=f'{chunk.source} - {chunk.section}',
            ),
        )
        return embedding_values(response)[0]

    def _embed_chunks(self, chunks: List[Chunk]) -> np.ndarray:
        vectors = [self._embed_document(chunk) for chunk in chunks]
        return normalize_vectors(np.array(vectors, dtype='float32'))

    def _embed_query(self, query: str) -> np.ndarray:
        response = embedding_client.models.embed_content(
            model=self.model,
            contents=query,
            config=types.EmbedContentConfig(task_type='RETRIEVAL_QUERY'),
        )
        return normalize_vectors(np.array(embedding_values(response), dtype='float32'))

    def _build_index(self, embeddings: np.ndarray):
        if self.faiss is None:
            return None
        index = self.faiss.IndexFlatIP(embeddings.shape[1])
        index.add(embeddings)
        return index

    def search(self, query: str, k: int = 6) -> List[Chunk]:
        query_vector = self._embed_query(query)
        k = min(k, len(self.chunks))

        if self.index is not None:
            scores, indices = self.index.search(query_vector, k)
            ranked = zip(indices[0], scores[0])
        else:
            scores = self.embeddings @ query_vector[0]
            indices = np.argsort(scores)[::-1][:k]
            ranked = ((idx, scores[idx]) for idx in indices)

        results: List[Chunk] = []
        for idx, score in ranked:
            if int(idx) < 0:
                continue
            results.append(replace(self.chunks[int(idx)], score=float(score)))
        return results


retriever = GeminiVectorRetriever(chunks)
retriever.search('severity 1 notification triage SLA', k=3)

[Chunk(id='cairnveil_security_policies:0021', source='cairnveil_security_policies.md', section='Customer Notification', text='## Customer Notification For severity 1 incidents, Cairnveil Security sends an initial customer notification within 30 minutes after triage confirms a credible incident. For severity 2 incidents, customer notification occurs within 1 hour after triage confirmation.', score=0.7501041889190674),
 Chunk(id='cairnveil_security_policies:0017', source='cairnveil_security_policies.md', section='Severity 1', text='### Severity 1 Severity 1 means active compromise, confirmed ransomware activity, business-critical cloud credential abuse, or attacker control of privileged identity. Initial triage begins within 15 minutes for ShieldGuard Pro and Enterprise customers.', score=0.7459653615951538),
 Chunk(id='cairnveil_security_policies:0019', source='cairnveil_security_policies.md', section='Severity 3', text='### Severity 3 Severity 3 means suspicious activity that requires 

## Kilo Gateway Helpers

The notebook uses Kilo's OpenAI-compatible gateway with `nvidia/nemotron-3-super-120b-a12b:free`. The JSON helper keeps graph decisions structured even when the model includes extra prose.

In [13]:
class RetrievalDecision(BaseModel):
    should_retrieve: bool = Field(..., description='Whether company documents are required.')
    reason: str = Field(..., description='Brief reason for the routing decision.')


class RelevanceDecision(BaseModel):
    is_relevant: bool = Field(..., description='Whether the chunk helps answer the question.')
    reason: str = Field(..., description='Brief reason for the relevance decision.')


class SupportDecision(BaseModel):
    support: Literal['fully_supported', 'partially_supported', 'unsupported']
    reason: str = Field(..., description='Brief evidence support critique.')


class UtilityDecision(BaseModel):
    utility: Literal['useful', 'not_useful']
    reason: str = Field(..., description='Brief usefulness critique.')


class RewriteDecision(BaseModel):
    retrieval_query: str = Field(..., description='Improved query for the local cybersecurity corpus.')
    reason: str = Field(..., description='Why this query should retrieve better evidence.')


def chat_text(system_prompt: str, user_prompt: str, *, temperature: float = 0.0, max_tokens: int = 900) -> str:
    if client is None:
        raise RuntimeError('Missing KILOCODE_API_KEY. Add it to .env, then rerun the setup cell.')

    response = client.chat.completions.create(
        model=KILO_MODEL,
        messages=[
            {'role': 'system', 'content': system_prompt},
            {'role': 'user', 'content': user_prompt},
        ],
        temperature=temperature,
        max_tokens=max_tokens,
    )
    return response.choices[0].message.content or ''


def iter_json_objects(text: str):
    """Yield every balanced top-level {...} substring in `text`, in order.

    Tracks JSON string boundaries so braces inside strings are ignored. The
    caller validates each candidate against a Pydantic model and keeps the
    first one that matches, which lets us skip a schema the LLM may have
    echoed before producing the real answer.
    """
    depth = 0
    start = -1
    in_string = False
    escape = False
    for i, ch in enumerate(text):
        if in_string:
            if escape:
                escape = False
            elif ch == '\\':
                escape = True
            elif ch == '"':
                in_string = False
            continue
        if ch == '"':
            in_string = True
        elif ch == '{':
            if depth == 0:
                start = i
            depth += 1
        elif ch == '}' and depth > 0:
            depth -= 1
            if depth == 0 and start >= 0:
                yield text[start:i + 1]
                start = -1


def call_json(model_cls: type[BaseModel], system_prompt: str, user_prompt: str, fallback: BaseModel) -> BaseModel:
    schema = json.dumps(model_cls.model_json_schema(), indent=2)
    json_user_prompt = f'''{user_prompt}

Return ONLY one JSON object that is an INSTANCE of this schema.
Do NOT repeat the schema. Do NOT include "properties" or "type": "object".
Schema (for reference, do not echo):
{schema}
'''
    try:
        raw = chat_text(system_prompt, json_user_prompt, max_tokens=600)
    except Exception as exc:
        print(f'JSON decision fallback for {model_cls.__name__}: {exc}')
        return fallback

    last_error = None
    for candidate in iter_json_objects(raw):
        try:
            return model_cls.model_validate_json(candidate)
        except Exception as exc:
            last_error = exc

    preview = raw[:200].replace('\n', ' ')
    print(f'JSON decision fallback for {model_cls.__name__}: no candidate validated ({last_error}); raw={preview!r}')
    return fallback


CYBER_ASSISTANT_SYSTEM = '''You are a careful cybersecurity RAG assistant for Cairnveil Security.
Be precise, cite only provided company facts when retrieval context is supplied, and avoid inventing policy, pricing, SLA, compliance, or incident-response details.'''

In [14]:
class SelfRAGState(TypedDict, total=False):
    question: str
    retrieval_query: str
    need_retrieval: bool
    retrieved_docs: List[Dict[str, Any]]
    relevant_docs: List[Dict[str, Any]]
    context: str
    answer: str
    support: str
    utility: str
    critique: str
    retries: int
    max_retries: int
    trace: List[str]


def chunk_to_dict(chunk: Chunk) -> Dict[str, Any]:
    return asdict(chunk)


def format_context(docs: List[Dict[str, Any]]) -> str:
    return '\n\n'.join(
        f'[Doc {i}] source={doc["source"]}; section={doc["section"]}; score={doc.get("score", 0):.3f}\n{doc["text"]}'
        for i, doc in enumerate(docs, start=1)
    )


def add_trace(state: SelfRAGState, message: str) -> List[str]:
    return list(state.get('trace', [])) + [message]

## Self-RAG Nodes

Each function below maps to one graph node: retrieval decision, retrieval, relevance grading, grounded generation, support critique, revision, usefulness critique, and query rewriting.

In [15]:
def decide_retrieval(state: SelfRAGState) -> Dict[str, Any]:
    fallback = RetrievalDecision(should_retrieve=True, reason='Default to retrieval for company-specific cybersecurity questions.')
    decision = call_json(
        RetrievalDecision,
        CYBER_ASSISTANT_SYSTEM,
        f'''Question: {state['question']}

Decide if internal Cairnveil Security documents are needed.
Return true for company-specific services, pricing, policies, SLAs, compliance support, incident playbooks, data handling, or evidence retention.
Return false only for generic cybersecurity concepts that do not require company facts.''',
        fallback,
    )
    return {
        'need_retrieval': decision.should_retrieve,
        'trace': add_trace(state, f'decide_retrieval: {decision.should_retrieve} ({decision.reason})'),
    }


def generate_direct(state: SelfRAGState) -> Dict[str, Any]:
    answer = chat_text(
        CYBER_ASSISTANT_SYSTEM,
        f'''Answer this as a general cybersecurity question without making claims about Cairnveil Security internal policies.

Question: {state['question']}''',
    )
    return {'answer': answer, 'trace': add_trace(state, 'generate_direct: answered without retrieval')}


def retrieve(state: SelfRAGState) -> Dict[str, Any]:
    query = state.get('retrieval_query') or state['question']
    hits = retriever.search(query, k=6)
    return {
        'retrieved_docs': [chunk_to_dict(hit) for hit in hits],
        'trace': add_trace(state, f'retrieve: {len(hits)} chunks for query={query!r}'),
    }


def grade_relevance(state: SelfRAGState) -> Dict[str, Any]:
    question = state['question']
    relevant_docs: List[Dict[str, Any]] = []

    for doc in state.get('retrieved_docs', []):
        fallback = RelevanceDecision(
            is_relevant=doc.get('score', 0.0) >= 0.25,
            reason='Fallback relevance based on Gemini embedding similarity score.',
        )
        decision = call_json(
            RelevanceDecision,
            CYBER_ASSISTANT_SYSTEM,
            f'''Question: {question}

Candidate document:
Source: {doc['source']}
Section: {doc['section']}
Text: {doc['text']}

Does this document contain facts that help answer the question?''',
            fallback,
        )
        if decision.is_relevant:
            doc = dict(doc)
            doc['relevance_reason'] = decision.reason
            relevant_docs.append(doc)

    context = format_context(relevant_docs)
    return {
        'relevant_docs': relevant_docs,
        'context': context,
        'trace': add_trace(state, f'grade_relevance: kept {len(relevant_docs)} chunks'),
    }


def no_relevant_docs(state: SelfRAGState) -> Dict[str, Any]:
    answer = 'I could not find enough support in the Cairnveil Security documents to answer that reliably.'
    return {'answer': answer, 'trace': add_trace(state, 'no_relevant_docs: stopped without grounded context')}


def generate_from_context(state: SelfRAGState) -> Dict[str, Any]:
    answer = chat_text(
        CYBER_ASSISTANT_SYSTEM,
        f'''Use ONLY the context below to answer the question.
If the context does not contain a fact, say that the documents do not specify it.
Cite facts with bracketed citations using source and section names.

Question: {state['question']}

Context:
{state.get('context', '')}''',
        max_tokens=1000,
    )
    return {'answer': answer, 'trace': add_trace(state, 'generate_from_context: drafted grounded answer')}


def grade_support(state: SelfRAGState) -> Dict[str, Any]:
    fallback = SupportDecision(support='partially_supported', reason='Fallback support grade.')
    decision = call_json(
        SupportDecision,
        CYBER_ASSISTANT_SYSTEM,
        f'''Question: {state['question']}

Context:
{state.get('context', '')}

Answer:
{state.get('answer', '')}

Grade whether every material claim in the answer is supported by the context.''',
        fallback,
    )
    return {
        'support': decision.support,
        'critique': decision.reason,
        'trace': add_trace(state, f'grade_support: {decision.support} ({decision.reason})'),
    }


def revise_answer(state: SelfRAGState) -> Dict[str, Any]:
    retries = int(state.get('retries', 0)) + 1
    revised = chat_text(
        CYBER_ASSISTANT_SYSTEM,
        f'''Revise the answer so every material claim is supported by the context.
Remove unsupported details. Keep citations.

Question: {state['question']}

Context:
{state.get('context', '')}

Critique:
{state.get('critique', '')}

Current answer:
{state.get('answer', '')}''',
        max_tokens=900,
    )
    return {
        'answer': revised,
        'retries': retries,
        'trace': add_trace(state, f'revise_answer: revision pass {retries}'),
    }


def grade_usefulness(state: SelfRAGState) -> Dict[str, Any]:
    fallback = UtilityDecision(utility='useful', reason='Fallback accepts the supported answer.')
    decision = call_json(
        UtilityDecision,
        CYBER_ASSISTANT_SYSTEM,
        f'''Question: {state['question']}

Answer:
{state.get('answer', '')}

Is this answer useful, concise, and responsive to the user?''',
        fallback,
    )
    return {
        'utility': decision.utility,
        'critique': decision.reason,
        'trace': add_trace(state, f'grade_usefulness: {decision.utility} ({decision.reason})'),
    }


def rewrite_query(state: SelfRAGState) -> Dict[str, Any]:
    retries = int(state.get('retries', 0)) + 1
    fallback = RewriteDecision(retrieval_query=state['question'], reason='Fallback keeps original question.')
    decision = call_json(
        RewriteDecision,
        CYBER_ASSISTANT_SYSTEM,
        f'''Original question: {state['question']}
Previous retrieval query: {state.get('retrieval_query') or state['question']}
Previous critique: {state.get('critique', '')}

Rewrite the query using concise cybersecurity and company-document keywords so local retrieval can find better evidence.''',
        fallback,
    )
    return {
        'retrieval_query': decision.retrieval_query,
        'retries': retries,
        'trace': add_trace(state, f'rewrite_query: {decision.retrieval_query!r} ({decision.reason})'),
    }


In [16]:
def retries_exhausted(state: SelfRAGState) -> bool:
    return int(state.get('retries', 0)) >= int(state.get('max_retries', 2))


def route_after_decide(state: SelfRAGState) -> Literal['generate_direct', 'retrieve']:
    return 'retrieve' if state.get('need_retrieval') else 'generate_direct'


def route_after_relevance(state: SelfRAGState) -> Literal['generate_from_context', 'rewrite_query', 'no_relevant_docs']:
    if state.get('relevant_docs'):
        return 'generate_from_context'
    return 'no_relevant_docs' if retries_exhausted(state) else 'rewrite_query'


def route_after_support(state: SelfRAGState) -> Literal['grade_usefulness', 'revise_answer']:
    if state.get('support') == 'fully_supported' or retries_exhausted(state):
        return 'grade_usefulness'
    return 'revise_answer'


def route_after_utility(state: SelfRAGState) -> Literal['rewrite_query', 'end']:
    if state.get('utility') == 'useful' or retries_exhausted(state):
        return 'end'
    return 'rewrite_query'


workflow = StateGraph(SelfRAGState)

workflow.add_node('decide_retrieval', decide_retrieval)
workflow.add_node('generate_direct', generate_direct)
workflow.add_node('retrieve', retrieve)
workflow.add_node('grade_relevance', grade_relevance)
workflow.add_node('no_relevant_docs', no_relevant_docs)
workflow.add_node('generate_from_context', generate_from_context)
workflow.add_node('grade_support', grade_support)
workflow.add_node('revise_answer', revise_answer)
workflow.add_node('grade_usefulness', grade_usefulness)
workflow.add_node('rewrite_query', rewrite_query)

workflow.add_edge(START, 'decide_retrieval')
workflow.add_conditional_edges('decide_retrieval', route_after_decide, {
    'generate_direct': 'generate_direct',
    'retrieve': 'retrieve',
})
workflow.add_edge('generate_direct', END)
workflow.add_edge('retrieve', 'grade_relevance')
workflow.add_conditional_edges('grade_relevance', route_after_relevance, {
    'generate_from_context': 'generate_from_context',
    'rewrite_query': 'rewrite_query',
    'no_relevant_docs': 'no_relevant_docs',
})
workflow.add_edge('no_relevant_docs', END)
workflow.add_edge('generate_from_context', 'grade_support')
workflow.add_conditional_edges('grade_support', route_after_support, {
    'grade_usefulness': 'grade_usefulness',
    'revise_answer': 'revise_answer',
})
workflow.add_edge('revise_answer', 'grade_support')
workflow.add_conditional_edges('grade_usefulness', route_after_utility, {
    'rewrite_query': 'rewrite_query',
    'end': END,
})
workflow.add_edge('rewrite_query', 'retrieve')

app = workflow.compile()
print('Self-RAG graph compiled.')

Self-RAG graph compiled.


In [17]:
def ask(question: str, *, max_retries: int = 2, show_trace: bool = True) -> SelfRAGState:
    initial_state: SelfRAGState = {
        'question': question,
        'retrieval_query': question,
        'need_retrieval': True,
        'retrieved_docs': [],
        'relevant_docs': [],
        'context': '',
        'answer': '',
        'support': '',
        'utility': '',
        'critique': '',
        'retries': 0,
        'max_retries': max_retries,
        'trace': [],
    }
    result = app.invoke(initial_state, config={'recursion_limit': 30})
    print(result.get('answer', ''))

    if show_trace:
        print('\nTrace:')
        for step in result.get('trace', []):
            print('-', step)
    return result


## Trialssss

In [18]:
# Company-specific: should retrieve.
result = ask('Which ShieldGuard plan includes monthly purple-team validation?')

The ShieldGuard Pro plan includes monthly purple‑team validation.【Doc 1 source=cairnveil_services_and_pricing.md section=ShieldGuard Pro】

Trace:
- decide_retrieval: True (The question asks about a specific ShieldGuard plan feature (monthly purple-team validation), which requires company-specific service details.)
- retrieve: 6 chunks for query='Which ShieldGuard plan includes monthly purple-team validation?'
- grade_relevance: kept 1 chunks
- generate_from_context: drafted grounded answer
- grade_support: fully_supported (The context explicitly states that ShieldGuard Pro includes monthly purple-team validation of the highest-risk detection gaps.)
- grade_usefulness: useful (Answer directly identifies the correct plan with citation, concise and responsive.)


In [19]:
# Company policy: should retrieve and cite evidence.
result = ask('What is the severity 1 customer notification policy?')

For severity 1 incidents, Cairnveil Security sends an initial customer notification within 30 minutes after triage confirms a credible incident【cairnveil_security_policies.md, Customer Notification】.

Trace:
- decide_retrieval: True (The question asks about Cairnveil Security's specific severity 1 customer notification policy, which requires internal company documents to answer accurately.)
- retrieve: 6 chunks for query='What is the severity 1 customer notification policy?'
- grade_relevance: kept 1 chunks
- generate_from_context: drafted grounded answer
- grade_support: fully_supported (The answer directly restates the policy statement from the provided context regarding severity 1 incident notification timing.)
- grade_usefulness: useful (The answer directly states the severity 1 customer notification policy (initial notification within 30 minutes after triage confirms a credible incident) and cites the source, making it useful, concise, and responsive.)


In [ ]:
# Incident playbook: may answer directly without retrieval.
result = ask('What evidence should analysts collect during a phishing investigation?')

During a phishing investigation, analysts should gather a variety of artifacts that can help determine the origin, scope, and impact of the attack. The following evidence types are commonly collected:

| Evidence Category | Specific Items to Collect | Why It Matters |
|-------------------|---------------------------|----------------|
| **Email Artifacts** | • Full raw email (including MIME parts) <br>• Original .eml or .msg file <br>• Email headers (Received, From, To, Subject, Date, Message-ID, DKIM/SPF/DMARC results) <br>• Attachments (preserve original files, compute hash values) | Headers reveal routing, spoofing, and authentication results; attachments may contain malware or malicious macros. |
| **URL / Link Evidence** | • All URLs embedded in the body (including obfuscated or shortened links) <br>• Screenshots of the landing page (if safe to view) <br>• DNS records (A, AAAA, CNAME, MX) for each domain <br>• WHOIS registration data <br>• SSL/TLS certificate details (issuer, valid

In [21]:
# Generic cybersecurity concept: may answer directly without retrieval.
result = ask('Explain phishing in simple terms for a new employee.')

Phishing is a type of online scam where attackers pretend to be a trusted person or organization—like a bank, a coworker, or a well‑known company—to trick you into giving away sensitive information (such as passwords, credit‑card numbers, or personal data) or into clicking a link or opening an attachment that installs malicious software.  

Typical signs of a phishing attempt include:

- Unexpected emails or messages that ask you to act quickly (“Your account will be locked!”)  
- Generic greetings (“Dear Customer”) instead of your name  
- Misspelled words, odd grammar, or suspicious email addresses/URLs  
- Requests for passwords, Social Security numbers, or other confidential data  
- Links that look similar to a legitimate site but have slight differences (e.g., “paypa1.com” instead of “paypal.com”)  

If you’re unsure, don’t click any links or open attachments. Instead, verify the request through a separate, trusted channel (like calling the organization using a known phone number